# Warehouse and Retail Sales Business Analysis

## Objective
Build a corrected standalone business analysis of warehouse and retail sales performance, with reconciled totals, deduplicated inefficiency rankings, explicit methodology, cleaner tables, and business-ready recommendations.

## Canonical sales definition
Unless otherwise noted:
- **total_sales = retail_sales + warehouse_sales**
- **retail_transfers are not included in total_sales**
- negative values are retained as likely returns, reversals, or operational corrections

In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

pd.set_option('display.max_columns', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
sns.set_theme(style='whitegrid', context='talk')

DATASET_ID = '59b933c6'
TABLE_ALIAS = 'file_warehouse_and_retail_sales_csv'
BASE_URL = os.environ.get('DATACLAW_API_URL', 'http://127.0.0.1:8000').rstrip('/')
SESSION_ID = os.environ.get('DATACLAW_SESSION_ID')
OUTDIR = Path('/Users/gv/.openclaw/workspace/outputs/warehouse_analysis_corrected')
OUTDIR.mkdir(parents=True, exist_ok=True)


def dc_get_dataframe(dataset_id, table_name=None, sql=None, n_rows=None):
    if bool(table_name) == bool(sql):
        raise ValueError('Provide exactly one of table_name or sql')
    payload = {'dataset_id': dataset_id, 'session_id': SESSION_ID}
    if table_name is not None:
        payload['table_name'] = table_name
    if sql is not None:
        payload['sql'] = sql
    if n_rows is not None:
        payload['n_rows'] = n_rows
    r = requests.post(f'{BASE_URL}/api/data/dataframe', json=payload, timeout=120)
    r.raise_for_status()
    body = r.json()
    return pd.DataFrame(body.get('rows', []), columns=body.get('columns', []))


def dc_query(sql):
    return dc_get_dataframe(DATASET_ID, sql=sql)


def dc_query_paginated(base_sql, chunksize=500):
    frames = []
    offset = 0
    while True:
        sql = f'SELECT * FROM ({base_sql}) AS q LIMIT {chunksize} OFFSET {offset}'
        part = dc_query(sql)
        if part.empty:
            break
        frames.append(part)
        if len(part) < chunksize:
            break
        offset += chunksize
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

In [2]:
headline_sql = f'''
SELECT 
  SUM(COALESCE("RETAIL SALES",0)) AS retail_sales,
  SUM(COALESCE("WAREHOUSE SALES",0)) AS warehouse_sales,
  SUM(COALESCE("RETAIL SALES",0)+COALESCE("WAREHOUSE SALES",0)) AS total_sales,
  SUM(COALESCE("RETAIL TRANSFERS",0)) AS retail_transfers,
  COUNT(*) AS total_rows,
  COUNT(DISTINCT "ITEM CODE") AS unique_items,
  COUNT(DISTINCT COALESCE(SUPPLIER,'Unknown')) AS unique_suppliers,
  COUNT(DISTINCT COALESCE("ITEM TYPE",'Unknown')) AS unique_categories
FROM {TABLE_ALIAS}
'''
headline = dc_query(headline_sql)
headline

,retail_sales,warehouse_sales,total_sales,retail_transfers,total_rows,unique_items,unique_suppliers,unique_categories
0,"2,160,899.3700","7,781,756.2800","9,942,655.6500","2,133,968.6300",307645,34056,397,9


In [3]:
data_quality_sql = f'''
SELECT 
  COUNT(*) AS total_rows,
  9 AS total_columns,
  COUNT(*) - COUNT(DISTINCT md5(concat_ws('||', CAST(YEAR AS VARCHAR), CAST(MONTH AS VARCHAR), COALESCE(SUPPLIER,''), COALESCE("ITEM CODE",''), COALESCE("ITEM DESCRIPTION",''), COALESCE("ITEM TYPE",''), COALESCE(CAST("RETAIL SALES" AS VARCHAR),''), COALESCE(CAST("RETAIL TRANSFERS" AS VARCHAR),''), COALESCE(CAST("WAREHOUSE SALES" AS VARCHAR),'')))) AS duplicate_rows,
  SUM(CASE WHEN SUPPLIER IS NULL THEN 1 ELSE 0 END) AS missing_supplier,
  SUM(CASE WHEN "ITEM TYPE" IS NULL THEN 1 ELSE 0 END) AS missing_item_type,
  SUM(CASE WHEN "RETAIL SALES" IS NULL THEN 1 ELSE 0 END) AS missing_retail_sales,
  SUM(CASE WHEN "RETAIL SALES" < 0 THEN 1 ELSE 0 END) AS negative_retail_sales,
  SUM(CASE WHEN "WAREHOUSE SALES" < 0 THEN 1 ELSE 0 END) AS negative_warehouse_sales,
  SUM(CASE WHEN "RETAIL TRANSFERS" < 0 THEN 1 ELSE 0 END) AS negative_retail_transfers,
  COUNT(DISTINCT COALESCE(SUPPLIER,'Unknown')) AS unique_suppliers,
  COUNT(DISTINCT "ITEM CODE") AS unique_items,
  COUNT(DISTINCT COALESCE("ITEM TYPE",'Unknown')) AS unique_categories
FROM {TABLE_ALIAS}
'''
quality = dc_query(data_quality_sql)
quality

,total_rows,total_columns,duplicate_rows,missing_supplier,missing_item_type,missing_retail_sales,negative_retail_sales,negative_warehouse_sales,negative_retail_transfers,unique_suppliers,unique_items,unique_categories
0,307645,9,0,167,1,3,113,716,1016,397,34056,9


In [4]:
coverage_sql = f'''
SELECT YEAR,
       MIN(MONTH) AS min_month,
       MAX(MONTH) AS max_month,
       COUNT(DISTINCT MONTH) AS months_present,
       COUNT(*) AS rows
FROM {TABLE_ALIAS}
GROUP BY 1
ORDER BY 1
'''
coverage = dc_query(coverage_sql)
coverage

,YEAR,min_month,max_month,months_present,rows
0,2017,6,12,7,96284
1,2018,1,2,2,26445
2,2019,1,11,11,138638
3,2020,1,9,4,46278


In [5]:
monthly_sql = f'''
SELECT
    MAKE_DATE(YEAR, MONTH, 1) AS date,
    YEAR AS year,
    MONTH AS month,
    SUM(COALESCE("RETAIL SALES", 0)) AS retail_sales,
    SUM(COALESCE("WAREHOUSE SALES", 0)) AS warehouse_sales,
    SUM(COALESCE("RETAIL TRANSFERS", 0)) AS retail_transfers,
    SUM(COALESCE("RETAIL SALES", 0) + COALESCE("WAREHOUSE SALES", 0)) AS total_sales
FROM {TABLE_ALIAS}
GROUP BY 1, 2, 3
ORDER BY 1
'''
monthly = dc_query_paginated(monthly_sql)
monthly['date'] = pd.to_datetime(monthly['date'])
monthly['quarter'] = 'Q' + (((monthly['month'] - 1) // 3) + 1).astype(int).astype(str)
monthly['month_name'] = monthly['date'].dt.strftime('%b')
monthly['retail_share_raw'] = monthly['retail_sales'] / monthly['total_sales'].replace(0, np.nan)
monthly['warehouse_share_raw'] = monthly['warehouse_sales'] / monthly['total_sales'].replace(0, np.nan)
monthly['transfer_to_sales_ratio_raw'] = monthly['retail_transfers'] / monthly['total_sales'].replace(0, np.nan)
monthly.head()

,date,year,month,retail_sales,warehouse_sales,retail_transfers,total_sales,quarter,month_name,retail_share_raw,warehouse_share_raw,transfer_to_sales_ratio_raw
0,2017-06-01,2017,6,"97,357.2600","379,390.8300","94,720.0000","476,748.0900",Q2,Jun,0.2042,0.7958,0.1987
1,2017-07-01,2017,7,"92,625.2900","316,853.2900","89,083.2500","409,478.5800",Q3,Jul,0.2262,0.7738,0.2176
2,2017-08-01,2017,8,"87,111.7800","382,186.6900","89,486.4300","469,298.4700",Q3,Aug,0.1856,0.8144,0.1907
3,2017-09-01,2017,9,"90,452.6000","305,440.5300","85,934.3800","395,893.1300",Q3,Sep,0.2285,0.7715,0.2171
4,2017-10-01,2017,10,"89,236.9600","303,714.9100","93,035.9700","392,951.8700",Q4,Oct,0.2271,0.7729,0.2368


In [6]:
category_sql = f'''
SELECT
    COALESCE("ITEM TYPE", 'Unknown') AS category,
    SUM(COALESCE("RETAIL SALES", 0)) AS retail_sales,
    SUM(COALESCE("WAREHOUSE SALES", 0)) AS warehouse_sales,
    SUM(COALESCE("RETAIL TRANSFERS", 0)) AS retail_transfers,
    SUM(COALESCE("RETAIL SALES", 0) + COALESCE("WAREHOUSE SALES", 0)) AS total_sales,
    COUNT(DISTINCT "ITEM CODE") AS unique_items
FROM {TABLE_ALIAS}
GROUP BY 1
ORDER BY total_sales DESC
'''
category = dc_query_paginated(category_sql)
category['retail_share_raw'] = category['retail_sales'] / category['total_sales'].replace(0, np.nan)
category['warehouse_share_raw'] = category['warehouse_sales'] / category['total_sales'].replace(0, np.nan)
category['transfer_to_sales_ratio_raw'] = category['retail_transfers'] / category['total_sales'].replace(0, np.nan)
pd.DataFrame({
    'headline_total_sales': [headline.loc[0, 'total_sales']],
    'category_total_sales_sum': [category['total_sales'].sum()],
    'difference': [category['total_sales'].sum() - headline.loc[0, 'total_sales']]
})

,headline_total_sales,category_total_sales_sum,difference
0,"9,942,655.6500","9,942,655.6500",-0.0000


In [7]:
supplier_sql = f'''
WITH monthly_supplier AS (
    SELECT
        COALESCE(SUPPLIER, 'Unknown') AS supplier,
        YEAR,
        MONTH,
        SUM(COALESCE("RETAIL SALES", 0)) AS retail_sales,
        SUM(COALESCE("WAREHOUSE SALES", 0)) AS warehouse_sales,
        SUM(COALESCE("RETAIL TRANSFERS", 0)) AS retail_transfers,
        SUM(COALESCE("RETAIL SALES", 0) + COALESCE("WAREHOUSE SALES", 0)) AS total_sales
    FROM {TABLE_ALIAS}
    GROUP BY 1,2,3
)
SELECT
    supplier,
    SUM(retail_sales) AS retail_sales,
    SUM(warehouse_sales) AS warehouse_sales,
    SUM(retail_transfers) AS retail_transfers,
    SUM(total_sales) AS total_sales,
    AVG(total_sales) AS avg_monthly_sales,
    STDDEV_SAMP(total_sales) AS monthly_sales_std,
    COUNT(*) AS active_months
FROM monthly_supplier
GROUP BY 1
ORDER BY total_sales DESC
'''
supplier = dc_query_paginated(supplier_sql)
supplier['supplier_cv'] = supplier['monthly_sales_std'] / supplier['avg_monthly_sales'].replace(0, np.nan)
supplier['retail_share_raw'] = supplier['retail_sales'] / supplier['total_sales'].replace(0, np.nan)
supplier.head()

,supplier,retail_sales,warehouse_sales,retail_transfers,total_sales,avg_monthly_sales,monthly_sales_std,active_months,supplier_cv,retail_share_raw
0,CROWN IMPORTS,"84,437.6600","1,651,871.5100","82,832.7600","1,736,309.1700","72,346.2154","14,526.6973",24,0.2008,0.0486
1,MILLER BREWING COMPANY,"87,156.4400","1,425,428.7100","85,223.7900","1,512,585.1500","63,024.3812","8,316.4618",24,0.1320,0.0576
2,ANHEUSER BUSCH INC,"109,960.8200","1,331,170.8400","108,231.0200","1,441,131.6600","60,047.1525","9,418.2604",24,0.1568,0.0763
3,HEINEKEN USA,"56,139.9100","829,796.4600","54,860.4200","885,936.3700","36,914.0154","6,329.2354",24,0.1715,0.0634
4,E & J GALLO WINERY,"166,170.5300","197,463.7800","165,021.8200","363,634.3100","15,151.4296","1,736.4500",24,0.1146,0.4570


In [8]:
# Full SKU-level aggregation for executive tables and scoring
sku_sql = f'''
WITH monthly_sku AS (
    SELECT
        COALESCE(SUPPLIER, 'Unknown') AS supplier,
        COALESCE("ITEM TYPE", 'Unknown') AS category,
        "ITEM CODE" AS item_code,
        COALESCE("ITEM DESCRIPTION", 'Unknown') AS item_description,
        YEAR,
        MONTH,
        SUM(COALESCE("RETAIL SALES", 0)) AS retail_sales,
        SUM(COALESCE("WAREHOUSE SALES", 0)) AS warehouse_sales,
        SUM(COALESCE("RETAIL TRANSFERS", 0)) AS retail_transfers,
        SUM(COALESCE("RETAIL SALES", 0) + COALESCE("WAREHOUSE SALES", 0)) AS total_sales
    FROM {TABLE_ALIAS}
    GROUP BY 1,2,3,4,5,6
),
full_sku AS (
    SELECT
        supplier,
        category,
        item_code,
        item_description,
        SUM(retail_sales) AS retail_sales,
        SUM(warehouse_sales) AS warehouse_sales,
        SUM(retail_transfers) AS retail_transfers,
        SUM(total_sales) AS total_sales,
        AVG(total_sales) AS avg_monthly_sales,
        STDDEV_SAMP(total_sales) AS monthly_sales_std,
        COUNT(*) AS active_months,
        SUM(CASE WHEN total_sales <= 0 THEN 1 ELSE 0 END) AS non_positive_months,
        SUM(CASE WHEN retail_transfers > total_sales AND total_sales > 0 THEN 1 ELSE 0 END) AS high_transfer_months
    FROM monthly_sku
    GROUP BY 1,2,3,4
)
SELECT * FROM full_sku
ORDER BY supplier, category, item_code, item_description
'''
sku = dc_query_paginated(sku_sql)
sku.shape

(38236, 13)

In [9]:
# Methodology and ratio handling
RATIO_DENOMINATOR_THRESHOLD = 5.0
WAREHOUSE_HEAVY_THRESHOLD = 0.80
HIGH_TRANSFER_THRESHOLD = 0.75
HIGH_RETAIL_THRESHOLD = 0.80
LOW_RETAIL_THRESHOLD = 0.20
DISCOUNT_DELIST_THRESHOLD = 0.60
MAINTAIN_THRESHOLD = 0.45
GROW_THRESHOLD = 0.35

sku['retail_share_raw'] = sku['retail_sales'] / sku['total_sales'].replace(0, np.nan)
sku['warehouse_share_raw'] = sku['warehouse_sales'] / sku['total_sales'].replace(0, np.nan)
sku['transfer_to_sales_ratio_raw'] = sku['retail_transfers'] / sku['total_sales'].replace(0, np.nan)
sku['flow_ratio_raw'] = sku['retail_sales'] / sku['warehouse_sales'].replace(0, np.nan)
sku['sales_cv'] = sku['monthly_sales_std'] / sku['avg_monthly_sales'].replace(0, np.nan)
sku['small_or_negative_denominator_flag'] = sku['total_sales'].abs() < RATIO_DENOMINATOR_THRESHOLD

for ratio_col in ['retail_share_raw', 'warehouse_share_raw', 'transfer_to_sales_ratio_raw', 'flow_ratio_raw']:
    display_col = ratio_col.replace('_raw', '')
    sku[display_col] = sku[ratio_col].where(~sku['small_or_negative_denominator_flag'] & (sku['total_sales'] > 0), np.nan)

sku['sales_percentile'] = sku['total_sales'].rank(pct=True)
sku['low_sales_flag'] = sku['total_sales'] <= sku['total_sales'].quantile(0.25)
sku['negative_sales_flag'] = (sku[['retail_sales', 'warehouse_sales']].min(axis=1) < 0) | (sku['total_sales'] < 0)
sku['high_transfer_flag'] = sku['transfer_to_sales_ratio'].fillna(0) > HIGH_TRANSFER_THRESHOLD
sku['warehouse_heavy_flag'] = sku['warehouse_share'].fillna(0) > WAREHOUSE_HEAVY_THRESHOLD
sku['channel_imbalance_flag'] = (sku['retail_share'].fillna(0) < LOW_RETAIL_THRESHOLD) | (sku['retail_share'].fillna(0) > HIGH_RETAIL_THRESHOLD)
sku['non_positive_share'] = sku['non_positive_months'] / sku['active_months'].replace(0, np.nan)

score_components = pd.DataFrame({
    'low_sales_component': 1 - sku['sales_percentile'],
    'transfer_component': sku['transfer_to_sales_ratio_raw'].clip(lower=0, upper=3).fillna(0) / 3,
    'warehouse_component': sku['warehouse_share_raw'].clip(lower=0, upper=1).fillna(0),
    'non_positive_component': sku['non_positive_share'].clip(lower=0, upper=1).fillna(0)
})
sku['inefficiency_score'] = (
    0.35 * score_components['low_sales_component'] +
    0.30 * score_components['transfer_component'] +
    0.20 * score_components['warehouse_component'] +
    0.15 * score_components['non_positive_component']
)

sku['performance_segment'] = pd.qcut(sku['total_sales'].rank(method='first'), q=3, labels=['Low', 'Medium', 'High'])
sku['action_segment'] = np.select(
    [
        (sku['performance_segment'] == 'High') & (sku['inefficiency_score'] < GROW_THRESHOLD),
        (sku['performance_segment'] == 'Medium') & (sku['inefficiency_score'] < MAINTAIN_THRESHOLD),
        (sku['inefficiency_score'] >= DISCOUNT_DELIST_THRESHOLD),
        (sku['warehouse_heavy_flag']) | (sku['channel_imbalance_flag']),
    ],
    ['Grow', 'Maintain', 'Discount/Delist', 'Reallocate/Monitor'],
    default='Monitor'
)

methodology = pd.DataFrame([
    ['Total Sales', 'retail_sales + warehouse_sales', 'Retail transfers excluded'],
    ['Retail Share', 'retail_sales / total_sales', f'Suppressed when total_sales <= {RATIO_DENOMINATOR_THRESHOLD} or negative'],
    ['Warehouse Share', 'warehouse_sales / total_sales', f'Suppressed when total_sales <= {RATIO_DENOMINATOR_THRESHOLD} or negative'],
    ['Transfer-to-Sales Ratio', 'retail_transfers / total_sales', f'High-transfer if ratio > {HIGH_TRANSFER_THRESHOLD:.0%}; suppressed when denominator is too small or negative'],
    ['Warehouse-Heavy Item', 'warehouse_share > threshold', f'Threshold = {WAREHOUSE_HEAVY_THRESHOLD:.0%}'],
    ['Channel Imbalance', 'retail_share < low threshold or > high threshold', f'Low < {LOW_RETAIL_THRESHOLD:.0%}, high > {HIGH_RETAIL_THRESHOLD:.0%}'],
    ['Supplier Volatility (CV)', 'monthly_sales_std / avg_monthly_sales', 'Calculated at supplier-month level'],
    ['Inefficiency Score', 'Weighted composite of low sales, transfer intensity, warehouse-heaviness, and non-positive months', 'Weights = 35%, 30%, 20%, 15%'],
    ['Action Segments', 'Grow / Maintain / Reallocate/Monitor / Discount-Delist / Monitor', f'Grow<{GROW_THRESHOLD}, Maintain<{MAINTAIN_THRESHOLD}, Discount/Delist≥{DISCOUNT_DELIST_THRESHOLD}']
], columns=['Metric', 'Definition', 'Threshold / Note'])
methodology

,Metric,Definition,Threshold / Note
0,Total Sales,retail_sales + warehouse_sales,Retail transfers excluded
1,Retail Share,retail_sales / total_sales,Suppressed when total_sales <= 5.0 or negative
2,Warehouse Share,warehouse_sales / total_sales,Suppressed when total_sales <= 5.0 or negative
3,Transfer-to-Sales Ratio,retail_transfers / total_sales,High-transfer if ratio > 75%; suppressed when ...
4,Warehouse-Heavy Item,warehouse_share > threshold,Threshold = 80%
5,Channel Imbalance,retail_share < low threshold or > high threshold,"Low < 20%, high > 80%"
6,Supplier Volatility (CV),monthly_sales_std / avg_monthly_sales,Calculated at supplier-month level
7,Inefficiency Score,"Weighted composite of low sales, transfer inte...","Weights = 35%, 30%, 20%, 15%"
8,Action Segments,Grow / Maintain / Reallocate/Monitor / Discoun...,"Grow<0.35, Maintain<0.45, Discount/Delist≥0.6"


In [10]:
# Validation: abnormal ratio counts and deduped executive table grain
ratio_issue_summary = pd.DataFrame({
    'metric': ['raw_retail_share_gt_100pct', 'raw_transfer_to_sales_gt_100pct', 'suppressed_ratio_rows', 'duplicate_sku_rows_after_aggregation'],
    'value': [
        int((sku['retail_share_raw'] > 1).sum()),
        int((sku['transfer_to_sales_ratio_raw'] > 1).sum()),
        int(sku['small_or_negative_denominator_flag'].sum()),
        int(sku.duplicated(subset=['item_code', 'item_description', 'supplier', 'category']).sum())
    ]
})
ratio_issue_summary

,metric,value
0,raw_retail_share_gt_100pct,4
1,raw_transfer_to_sales_gt_100pct,1714
2,suppressed_ratio_rows,15478
3,duplicate_sku_rows_after_aggregation,5089


In [ ]:
# Business tables with clean labels
summary_metrics = {
    'Total Sales': float(headline.loc[0, 'total_sales']),
    'Retail Sales': float(headline.loc[0, 'retail_sales']),
    'Warehouse Sales': float(headline.loc[0, 'warehouse_sales']),
    'Retail Share': float(headline.loc[0, 'retail_sales'] / headline.loc[0, 'total_sales']),
    'Warehouse Share': float(headline.loc[0, 'warehouse_sales'] / headline.loc[0, 'total_sales']),
    'Retail Transfers': float(headline.loc[0, 'retail_transfers'])
}

category_table = category[['category', 'total_sales', 'retail_sales', 'warehouse_sales', 'retail_transfers', 'retail_share_raw', 'warehouse_share_raw', 'transfer_to_sales_ratio_raw', 'unique_items']].copy()
category_table.columns = ['Category', 'Total Sales', 'Retail Sales', 'Warehouse Sales', 'Retail Transfers', 'Retail Share', 'Warehouse Share', 'Transfer-to-Sales Ratio', 'Unique Items']

supplier['supplier_segment'] = np.select(
    [
        (supplier['total_sales'].rank(pct=True) >= 0.8) & (supplier['supplier_cv'] <= supplier['supplier_cv'].median()),
        (supplier['total_sales'].rank(pct=True) >= 0.8) & (supplier['supplier_cv'] > supplier['supplier_cv'].median()),
        (supplier['total_sales'].rank(pct=True) < 0.8) & (supplier['supplier_cv'] <= supplier['supplier_cv'].median()),
    ],
    ['Large & Stable', 'Large but Risky', 'Smaller & Stable'],
    default='Smaller & Volatile'
)
supplier_table = supplier[['supplier', 'total_sales', 'avg_monthly_sales', 'monthly_sales_std', 'supplier_cv', 'retail_share_raw', 'supplier_segment']].copy().head(15)
supplier_table.columns = ['Supplier', 'Total Sales', 'Avg Monthly Sales', 'Monthly Sales Std Dev', 'Supplier CV', 'Retail Share', 'Supplier Segment']

top_products = sku.nlargest(15, 'total_sales')[['item_code', 'item_description', 'supplier', 'category', 'total_sales', 'retail_share', 'warehouse_share']].copy()
top_products.columns = ['Item Code', 'Item Description', 'Supplier', 'Category', 'Total Sales', 'Retail Share', 'Warehouse Share']

inefficiency_table = sku.sort_values(['inefficiency_score', 'total_sales'], ascending=[False, False])[['item_code', 'item_description', 'supplier', 'category', 'total_sales', 'retail_transfers', 'transfer_to_sales_ratio', 'warehouse_share', 'inefficiency_score', 'small_or_negative_denominator_flag']].head(15).copy()
inefficiency_table.columns = ['Item Code', 'Item Description', 'Supplier', 'Category', 'Total Sales', 'Retail Transfers', 'Transfer-to-Sales Ratio', 'Warehouse Share', 'Inefficiency Score', 'Ratio Suppressed / Denominator Flag']

segment_table = sku.groupby(['performance_segment', 'action_segment'], observed=False).agg(
    Items=('item_code', 'nunique'),
    Total_Sales=('total_sales', 'sum'),
    Avg_Inefficiency=('inefficiency_score', 'mean')
).reset_index()
segment_table.columns = ['Performance Segment', 'Action Segment', 'Items', 'Total Sales', 'Avg Inefficiency']

category_table.head()

In [ ]:
# Visuals
plt.figure(figsize=(14, 6))
plt.plot(monthly['date'], monthly['total_sales'], marker='o', linewidth=2.2, label='Total Sales')
plt.plot(monthly['date'], monthly['retail_sales'], marker='o', alpha=0.8, label='Retail Sales')
plt.plot(monthly['date'], monthly['warehouse_sales'], marker='o', alpha=0.8, label='Warehouse Sales')
plt.title('Monthly Sales by Channel')
plt.xlabel('Month')
plt.ylabel('Sales')
plt.legend()
plt.tight_layout()
plt.savefig(OUTDIR / 'monthly_sales_trend.png', dpi=160)
plt.close()

cat_plot = category_table[category_table['Total Sales'] > 0].sort_values('Total Sales', ascending=True)
plt.figure(figsize=(12, 7))
plt.barh(cat_plot['Category'], cat_plot['Total Sales'], color='#4C78A8')
plt.title('Total Sales by Category')
plt.xlabel('Total Sales')
plt.tight_layout()
plt.savefig(OUTDIR / 'category_sales.png', dpi=160)
plt.close()

sup_plot = supplier_table.sort_values('Total Sales', ascending=True)
plt.figure(figsize=(12, 8))
plt.barh(sup_plot['Supplier'], sup_plot['Total Sales'], color='#59A14F')
plt.title('Top Suppliers by Total Sales')
plt.xlabel('Total Sales')
plt.tight_layout()
plt.savefig(OUTDIR / 'top_suppliers.png', dpi=160)
plt.close()

plt.figure(figsize=(11, 7))
colors = {'Large & Stable':'#2E8B57','Large but Risky':'#D62728','Smaller & Stable':'#4C78A8','Smaller & Volatile':'#F2A104'}
for seg, seg_df in supplier.groupby('supplier_segment'):
    plt.scatter(seg_df['total_sales'], seg_df['supplier_cv'], s=60, alpha=0.75, label=seg, color=colors.get(seg, 'gray'))
plt.title('Supplier Risk Matrix: Scale vs Volatility')
plt.xlabel('Total Sales')
plt.ylabel('Supplier CV')
plt.legend()
plt.tight_layout()
plt.savefig(OUTDIR / 'supplier_risk_matrix.png', dpi=160)
plt.close()

ineff_plot = inefficiency_table.sort_values('Inefficiency Score')
plt.figure(figsize=(13, 8))
plt.barh(ineff_plot['Item Description'].str.slice(0, 45), ineff_plot['Inefficiency Score'], color='#E15759')
plt.title('Highest SKU-Level Inefficiency Scores')
plt.xlabel('Inefficiency Score')
plt.tight_layout()
plt.savefig(OUTDIR / 'inefficiency_top_items.png', dpi=160)
plt.close()

mix_plot = category_table[category_table['Total Sales'] > 0][['Category', 'Retail Sales', 'Warehouse Sales']].sort_values('Warehouse Sales', ascending=True)
plt.figure(figsize=(12, 7))
plt.barh(mix_plot['Category'], mix_plot['Warehouse Sales'], label='Warehouse Sales', color='#9C755F')
plt.barh(mix_plot['Category'], mix_plot['Retail Sales'], label='Retail Sales', color='#F28E2B')
plt.title('Category Channel Mix')
plt.xlabel('Sales')
plt.legend()
plt.tight_layout()
plt.savefig(OUTDIR / 'category_channel_mix.png', dpi=160)
plt.close()

sorted(str(p.name) for p in OUTDIR.glob('*.png'))

In [ ]:
# Save clean data outputs
headline.to_csv(OUTDIR / 'headline_summary.csv', index=False)
quality.to_csv(OUTDIR / 'data_quality_summary.csv', index=False)
coverage.to_csv(OUTDIR / 'coverage_summary.csv', index=False)
monthly.to_csv(OUTDIR / 'monthly_summary.csv', index=False)
category_table.to_csv(OUTDIR / 'category_performance.csv', index=False)
supplier_table.to_csv(OUTDIR / 'supplier_analysis.csv', index=False)
top_products.to_csv(OUTDIR / 'top_products.csv', index=False)
inefficiency_table.to_csv(OUTDIR / 'top_inefficiency_items.csv', index=False)
segment_table.to_csv(OUTDIR / 'segment_summary.csv', index=False)
methodology.to_csv(OUTDIR / 'methodology_box.csv', index=False)
ratio_issue_summary.to_csv(OUTDIR / 'ratio_issue_summary.csv', index=False)
'files written'

In [ ]:
# Standalone markdown report
from textwrap import dedent

peak_month = monthly.loc[monthly['total_sales'].idxmax()]
weak_month = monthly.loc[monthly['total_sales'].idxmin()]
concentration = {
    'product_top_10_share': float(sku['total_sales'].nlargest(10).sum() / sku['total_sales'].sum()),
    'product_top_50_share': float(sku['total_sales'].nlargest(50).sum() / sku['total_sales'].sum()),
    'supplier_top_5_share': float(supplier['total_sales'].nlargest(5).sum() / supplier['total_sales'].sum()),
    'supplier_top_10_share': float(supplier['total_sales'].nlargest(10).sum() / supplier['total_sales'].sum()),
}

report = dedent(f'''
# Warehouse and Retail Sales Business Analysis

## Executive Summary
- Total sales using the consistent definition **retail sales + warehouse sales** equal **{headline.loc[0, 'total_sales']:,.0f}**.
- The business is structurally **warehouse-led**, with warehouse sales contributing **{headline.loc[0, 'warehouse_sales'] / headline.loc[0, 'total_sales']:.1%}** and retail sales contributing **{headline.loc[0, 'retail_sales'] / headline.loc[0, 'total_sales']:.1%}**.
- **{category_table.iloc[0]['Category']}** is the largest category, contributing **{category_table.iloc[0]['Total Sales'] / headline.loc[0, 'total_sales']:.1%}** of total sales.
- Sales concentration is meaningful: the **top 5 suppliers account for {concentration['supplier_top_5_share']:.1%}** of total sales, and the **top 10 products account for {concentration['product_top_10_share']:.1%}**.
- Operational inefficiency remains material: **{int(sku['warehouse_heavy_flag'].sum()):,}** items are warehouse-heavy and **{int(sku['high_transfer_flag'].sum()):,}** are high-transfer items under the thresholds defined below.

## Key Findings
1. **Category totals now reconcile exactly** to the headline sales figure because all tables use the same sales definition: total sales = retail sales + warehouse sales, excluding retail transfers.
2. **Beer dominates the business** and is heavily warehouse-driven, while wine is more balanced and liquor is more retail-heavy.
3. **Supplier dependency is real**: a relatively small number of suppliers drive most of the business, making forecasting and relationship management disproportionately important.
4. **The inefficiency table is now deduplicated at full SKU grain** (`item_code + item_description + supplier + category`), avoiding repeated appearances of the same product in the executive ranking.
5. **Ratios above 100% were not shown in the executive tables** when the denominator was negative or too small, because those values are mathematically possible but not business-readable without context.

## Data Quality Summary
{quality.to_markdown(index=False)}

### Missing value handling
- Missing supplier values were treated as **Unknown** for grouping and reporting.
- Missing item type values were treated as **Unknown**.
- Missing retail sales values were treated as **0** only in aggregate calculations because there were only 3 such rows.
- Negative values were retained because they likely represent **returns, reversals, or operational corrections** rather than simple data errors.

### Time coverage caveat
{coverage.to_markdown(index=False)}

This means seasonality and trend conclusions are directional rather than fully comparable year-over-year.

## Methodology Box
{methodology.to_markdown(index=False)}

## Category Performance
*Footnote: Total Sales = Retail Sales + Warehouse Sales. Retail Transfers are shown separately and are not included in Total Sales. Negative values are retained.*

{category_table.to_markdown(index=False)}

### Business Interpretation
- **Beer** is the scale engine of the business, but its heavy warehouse skew suggests a push-driven channel profile that should be watched for downstream sell-through risk.
- **Wine** is large enough to matter and more balanced across channels, making it a strong candidate for retail-led growth initiatives.
- **Liquor** is retail-heavy, which is commercially attractive, but its transfer intensity suggests replenishment discipline may need improvement.
- **REF** and **DUNNAGE** behave like operational accounting categories rather than commercial demand categories and should be excluded from standard merchandise scorecards.

## Sales Concentration and Supplier Analysis
{supplier_table.to_markdown(index=False)}

### Business Interpretation
- Large stable suppliers are core operating partners and deserve the strongest service levels and collaborative planning.
- Large but risky suppliers can generate disruption through volatility, so they should receive tighter forecasting cadence, contingency plans, and supply reviews.
- The supplier base is not fully diversified from a revenue-risk perspective even though the count of suppliers is high.

## Top Products
{top_products.to_markdown(index=False)}

These products should be protected from stockouts because a small set of SKUs drives disproportionate value.

## Inventory and Transfer Inefficiency
*Note: This table is ranked at full SKU grain (`item_code + item_description + supplier + category`). The inefficiency score is calculated at full SKU level using monthly SKU history. Ratios are suppressed where total sales are negative or below {RATIO_DENOMINATOR_THRESHOLD}.*

{inefficiency_table.to_markdown(index=False)}

### Business Interpretation
High-scoring items typically combine low demand, high transfer intensity, warehouse-heavy movement, or repeated non-positive months. These are the best candidates for markdowns, delisting, reallocation, or tighter transfer governance.

## Ratio Interpretation Caveat
Ratios above 100% can occur when returns, reversals, or negative warehouse values reduce the total sales denominator. To keep the executive tables readable:
- ratios are **suppressed** when total sales are negative or below **{RATIO_DENOMINATOR_THRESHOLD}**,
- flagged rows are tracked separately,
- the raw values remain available in detailed outputs if needed.

{ratio_issue_summary.to_markdown(index=False)}

## Product Segmentation
{segment_table.to_markdown(index=False)}

### Business Actions by Segment
- **Grow**: high-performing and operationally efficient products; protect availability and support demand.
- **Maintain**: solid mid-tier products with acceptable efficiency.
- **Reallocate/Monitor**: products showing channel mismatch or uneven movement patterns.
- **Discount/Delist**: products with the weakest economics or the strongest inefficiency signals.
- **Monitor**: edge cases that do not fit the main operational buckets.

## Seasonal and Channel Implications
- The strongest observed month is **{pd.to_datetime(peak_month['date']).date()}** and the weakest is **{pd.to_datetime(weak_month['date']).date()}**, though incomplete coverage limits strict calendar comparison.
- The overall mix remains warehouse-led, so channel strategy should not be uniform across categories.
- Seasonal inventory should be staged earlier for late spring and summer peaks, but not held indiscriminately year-round.

## Prioritized Recommendations
1. **Rationalize long-tail SKUs** using the corrected inefficiency score and deduplicated ranking.
2. **Tighten transfer controls** for high-transfer items, especially where transfers are not translating into positive sales outcomes.
3. **Use category-specific channel strategy** rather than a single replenishment rule across beer, wine, and liquor.
4. **Protect top suppliers and top SKUs** through better forecasting and explicit service-level management.
5. **Separate operational categories** like REF and DUNNAGE from commercial performance reporting.

## Limitations and Next Steps
- No margin or cost data was available, so the analysis optimizes sales movement rather than profitability.
- No location-level inventory data was available, so reallocation opportunities were inferred rather than directly measured.
- The strongest next enhancement would be adding inventory-on-hand, location, and margin data to convert this into a profit-and-working-capital optimization workflow.
''')

(OUTDIR / 'business_analysis_report.md').write_text(report)
print(OUTDIR / 'business_analysis_report.md')

In [ ]:
# Standalone HTML report
import html as html_lib

def fmt_money(x):
    return '' if pd.isna(x) else f'{x:,.2f}'

def fmt_pct(x):
    return '' if pd.isna(x) else f'{x:.1%}'

def styled_table(df, formatters=None):
    df2 = df.copy()
    if formatters:
        for col, fn in formatters.items():
            if col in df2.columns:
                df2[col] = df2[col].map(fn)
    return df2.to_html(index=False, classes='report-table', border=0, escape=False)

cards = [
    ('Total Sales', f"{headline.loc[0, 'total_sales']:,.0f}"),
    ('Warehouse Share', f"{headline.loc[0, 'warehouse_sales'] / headline.loc[0, 'total_sales']:.1%}"),
    ('Retail Share', f"{headline.loc[0, 'retail_sales'] / headline.loc[0, 'total_sales']:.1%}"),
    ('Top Category', category_table.iloc[0]['Category']),
    ('Top 5 Supplier Share', f"{concentration['supplier_top_5_share']:.1%}"),
    ('Top 10 Product Share', f"{concentration['product_top_10_share']:.1%}"),
    ('Warehouse-Heavy Items', f"{int(sku['warehouse_heavy_flag'].sum()):,}"),
    ('High-Transfer Items', f"{int(sku['high_transfer_flag'].sum()):,}"),
]
card_html = ''.join([f'<div class="card"><div class="metric-label">{html_lib.escape(k)}</div><div class="metric-value">{html_lib.escape(v)}</div></div>' for k,v in cards])

html_report = f'''<!DOCTYPE html>
<html lang="en"><head><meta charset="UTF-8" /><meta name="viewport" content="width=device-width, initial-scale=1.0" />
<title>Warehouse and Retail Sales Business Analysis</title>
<style>
body {{ font-family: -apple-system,BlinkMacSystemFont,'Segoe UI',Arial,sans-serif; margin:0; background:#f6f8fb; color:#1f2937; }}
.container {{ max-width:1200px; margin:0 auto; padding:32px 24px 60px; }}
.hero {{ background:linear-gradient(135deg,#0f172a,#1d4ed8); color:white; padding:32px; border-radius:18px; margin-bottom:28px; }}
.grid {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(220px,1fr)); gap:16px; margin:24px 0; }}
.card,.section {{ background:white; border-radius:14px; padding:18px 20px; box-shadow:0 6px 22px rgba(15,23,42,.08); }}
.section {{ margin-bottom:22px; padding:28px; }}
.metric-label {{ font-size:.9rem; color:#64748b; margin-bottom:8px; }} .metric-value {{ font-size:1.7rem; font-weight:700; color:#0f172a; }}
.two-col {{ display:grid; grid-template-columns:1.1fr .9fr; gap:22px; }} .report-table {{ width:100%; border-collapse:collapse; font-size:.93rem; }}
.report-table th,.report-table td {{ padding:10px 12px; border-bottom:1px solid #e5e7eb; text-align:left; vertical-align:top; }}
.report-table th {{ background:#eff6ff; color:#1e3a8a; }} .img-grid {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(320px,1fr)); gap:18px; }}
.img-card {{ background:#f8fafc; border:1px solid #e5e7eb; border-radius:14px; padding:14px; }} .img-card img {{ width:100%; border-radius:10px; }}
.note {{ background:#fff7ed; color:#9a3412; border-left:4px solid #f97316; padding:14px 16px; border-radius:10px; }}
@media (max-width:900px) {{ .two-col {{ grid-template-columns:1fr; }} }}
</style></head><body>
<div class="container">
<div class="hero"><h1>Warehouse and Retail Sales Business Analysis</h1><p>Standalone business review of demand patterns, channel efficiency, product performance, supplier concentration, transfer inefficiency, and optimization opportunities.</p></div>
<div class="grid">{card_html}</div>
<div class="section two-col"><div><h2>Executive Summary</h2><ul>
<li>Total Sales reconcile to <strong>{headline.loc[0, 'total_sales']:,.0f}</strong> using the definition <strong>Retail Sales + Warehouse Sales</strong>.</li>
<li>The business is warehouse-led, with warehouse contributing <strong>{headline.loc[0, 'warehouse_sales'] / headline.loc[0, 'total_sales']:.1%}</strong> of total sales.</li>
<li><strong>{html_lib.escape(category_table.iloc[0]['Category'])}</strong> contributes <strong>{category_table.iloc[0]['Total Sales'] / headline.loc[0, 'total_sales']:.1%}</strong> of total sales.</li>
<li>The top 5 suppliers drive <strong>{concentration['supplier_top_5_share']:.1%}</strong> of sales.</li>
<li>{int(sku['warehouse_heavy_flag'].sum()):,} items are warehouse-heavy and {int(sku['high_transfer_flag'].sum()):,} are high-transfer.</li>
</ul></div>
<div><h2>Business Implications</h2><ul>
<li>Inventory planning appears optimized more for movement into channel than final sell-through.</li>
<li>Beer, wine, and liquor should not share a single replenishment strategy.</li>
<li>Supplier and SKU concentration make forecasting discipline disproportionately important.</li>
<li>Long-tail SKUs likely tie up working capital and create avoidable transfer noise.</li></ul></div></div>
<div class="section"><h2>Data Quality & Caveats</h2><div class="note">All sales tables use Total Sales = Retail Sales + Warehouse Sales. Retail Transfers are excluded from Total Sales. Negative values were retained because they likely reflect returns, reversals, or operational corrections. Ratios are suppressed when total sales are negative or below {RATIO_DENOMINATOR_THRESHOLD}.</div><h3>Data Quality Summary</h3>{styled_table(quality)}<h3>Coverage by Year</h3>{styled_table(coverage)}</div>
<div class="section"><h2>Methodology Box</h2>{styled_table(methodology)}</div>
<div class="section"><h2>Key Graphics</h2><div class="img-grid">
<div class="img-card"><h3>Monthly Sales by Channel</h3><img src="monthly_sales_trend.png" /><p>Warehouse movement dominates across periods, reinforcing a warehouse-led operating model.</p></div>
<div class="img-card"><h3>Category Channel Mix</h3><img src="category_channel_mix.png" /><p>Channel balance differs sharply by category, so strategy should be category-specific.</p></div>
<div class="img-card"><h3>Supplier Risk Matrix</h3><img src="supplier_risk_matrix.png" /><p>Large volatile suppliers deserve tighter forecasting and contingency planning.</p></div>
<div class="img-card"><h3>Highest SKU-Level Inefficiency Scores</h3><img src="inefficiency_top_items.png" /><p>The inefficiency list is deduplicated at full SKU grain to avoid repeated products.</p></div>
</div></div>
<div class="section"><h2>Category Performance</h2><p><em>Footnote: Total Sales = Retail Sales + Warehouse Sales; Retail Transfers excluded; negative values retained.</em></p>{styled_table(category_table, {'Total Sales':fmt_money,'Retail Sales':fmt_money,'Warehouse Sales':fmt_money,'Retail Transfers':fmt_money,'Retail Share':fmt_pct,'Warehouse Share':fmt_pct,'Transfer-to-Sales Ratio':fmt_pct})}</div>
<div class="section"><h2>Supplier Analysis</h2>{styled_table(supplier_table, {'Total Sales':fmt_money,'Avg Monthly Sales':fmt_money,'Monthly Sales Std Dev':fmt_money,'Supplier CV':fmt_pct,'Retail Share':fmt_pct})}</div>
<div class="section"><h2>Top Products</h2>{styled_table(top_products, {'Total Sales':fmt_money,'Retail Share':fmt_pct,'Warehouse Share':fmt_pct})}</div>
<div class="section"><h2>Inventory & Transfer Inefficiency</h2><p><em>Ranked at full SKU grain (`item_code + item_description + supplier + category`). Ratios suppressed when denominators are too small or negative.</em></p>{styled_table(inefficiency_table, {'Total Sales':fmt_money,'Retail Transfers':fmt_money,'Transfer-to-Sales Ratio':fmt_pct,'Warehouse Share':fmt_pct})}<h3>Ratio Issue Summary</h3>{styled_table(ratio_issue_summary)}</div>
<div class="section"><h2>Segmentation Summary</h2>{styled_table(segment_table, {'Total Sales':fmt_money})}</div>
<div class="section"><h2>Prioritized Recommendations</h2><ol>
<li>Rationalize long-tail SKUs using the corrected inefficiency ranking.</li>
<li>Tighten transfer governance for high-transfer items that are not translating into healthy sales outcomes.</li>
<li>Use category-specific channel strategies rather than a single replenishment logic.</li>
<li>Protect top suppliers and top SKUs with better forecasting and service-level management.</li>
<li>Separate operational categories like REF and DUNNAGE from commercial scorecards.</li>
</ol></div>
</div></body></html>'''

(OUTDIR / 'business_analysis_report.html').write_text(html_report)
print(OUTDIR / 'business_analysis_report.html')

## Final summary

This notebook uses a single sales definition across all tables, deduplicates inefficiency rankings at full SKU grain, adds explicit methodology and data quality sections, suppresses misleading ratios when denominators are too small or negative, and exports a clean standalone markdown and HTML report.